# APRENDIZAJE AUTOMÁTICO: DESCRIPCIÓN Y ORIGEN DEL DATASET
-------------------------
## Proyecto: Instancia de parcial  
## Profesor: Caballero Nicolás  
## Alumno: Calisaya Débora  
--------------

### Trabajo Práctico: Entrega 2 y Desarrollo del Modelo
### **Título:** Modelado predictivo de la demanda eléctrica y simulación de escenarios críticos en el sistema energético aislado de Tierra del Fuego.


Importación de Librerías

In [22]:
# Importación de librerías esenciales para manipulación de datos
import pandas as pd
import numpy as np

# Librerías para visualización y Análisis Exploratorio de Datos (EDA)
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración del entorno
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

print("Librerías base cargadas exitosamente.")

Librerías base cargadas exitosamente.


### Informe sobre el origen del dataset, de dónde provienen los datos (fuente, fecha de adquisición y preprocesamiento realizado)."

**Respuesta y Justificación:**
* **Fuente de los datos:** Los datos provienen del **Instituto Provincial de Análisis Estadístico y Censos (IPIEC)** de la Provincia de Tierra del Fuego, Antártida e Islas del Atlántico Sur. Se extrajeron de las series estadísticas de *Estadísticas Económicas - Energía Eléctrica*.
* **Portal Oficial:** [https://ipiec.tierradelfuego.gob.ar/estadisticas-economicas-2/](https://ipiec.tierradelfuego.gob.ar/estadisticas-economicas-2/)
* **Fecha de Adquisición:** Junio de 2026.
* **Proceso de Recopilación y Desafío Técnico:** Los datos se encuentran en un único archivo Excel (`15_3_01_Energia_electrica_consumida_por_tipo_usuario-1.xlsx`). La estructura de origen presenta la columna **"Período"** dividida jerárquicamente: el año se declara una sola vez al inicio de cada ciclo anual y los meses se listan consecutivamente en las filas inferiores de la misma columna. 

Para consolidar esto en Machine Learning, implementaremos una función ETL en Python que detecte los años, aplique una técnica de relleno hacia abajo (`ffill`), normalice las cadenas de texto de los meses a valores numéricos y unifique las pestañas de las localidades en una única matriz cronológica limpia.

Carga del Dataset

In [27]:
# Nombre del archivo original de Excel en tu carpeta
archivo_excel = '15_3_01_Energia_electrica_consumida_por_tipo_usuario-1.xlsx'

def procesar_hoja_ipiec(nombre_hoja, nombre_columna_total):
    # 1. Leemos el Excel saltando las 3 filas superiores de títulos (el encabezado real está en la fila 4)
    # Usamos usecols=[0, 1, 2] para tomar SOLO Período (Año), la columna sin nombre (Mes) y el Total (kWh)
    df_hoja = pd.read_excel(archivo_excel, sheet_name=nombre_hoja, skiprows=3, usecols=[0, 1, 2], engine='openpyxl')
    
    # 2. Renombramos de forma fija y directa las 3 columnas para evitar el IndexError
    df_hoja.columns = ['Anio_Raw', 'Mes_Raw', nombre_columna_total]
    
    # 3. Limpiamos espacios invisibles en los textos
    df_hoja['Anio_Raw'] = df_hoja['Anio_Raw'].astype(str).str.strip()
    df_hoja['Mes_Raw'] = df_hoja['Mes_Raw'].astype(str).str.strip()
    
    # 4. Reemplazamos celdas vacías por NaN para poder propagar el año hacia abajo
    df_hoja['Anio_Raw'] = df_hoja['Anio_Raw'].replace(['nan', 'None', ''], np.nan)
    
    # Filtramos filas que no contengan meses válidos o que sean basura
    df_hoja = df_hoja[df_hoja['Mes_Raw'].notna() & (df_hoja['Mes_Raw'] != 'nan')].reset_index(drop=True)
    
    # 5. Aplicamos Forward Fill (.ffill()) para que el Año se repita en sus 12 meses inferiores
    df_hoja['Anio_Raw'] = df_hoja['Anio_Raw'].ffill()
    
    # Diccionario para mapear los meses de texto a números
    meses_dict = {
        'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4, 'mayo': 5, 'junio': 6,
        'julio': 7, 'agosto': 8, 'septiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12,
        'noviembre ': 11, 'junio ': 6
    }
    
    fechas_limpias = []
    for idx, row in df_hoja.iterrows():
        try:
            # Quitamos los decimales del año (ej: de '2010.0' a 2010)
            anio_val = int(float(row['Anio_Raw']))
            mes_str = row['Mes_Raw'].lower()
            mes_val = meses_dict.get(mes_str, None)
            
            if mes_val is not None:
                fechas_limpias.append(pd.to_datetime(f"{anio_val}-{mes_val:02d}-01"))
            else:
                fechas_limpias.append(pd.NaT)
        except:
            fechas_limpias.append(pd.NaT)
            
    df_hoja['Fecha'] = fechas_limpias
    
    # Eliminamos registros que no pudieron convertirse en fechas reales (totales anuales o notas)
    df_hoja = df_hoja.dropna(subset=['Fecha']).reset_index(drop=True)
    
    return df_hoja[['Fecha', nombre_columna_total]]

# Ejecutamos la carga leyendo las 4 pestañas del Excel del IPIEC
df_total_prov = procesar_hoja_ipiec('Total TDF', 'Consumo_Total_Provincial')
df_ush = procesar_hoja_ipiec('Ushuaia', 'Consumo_Ushuaia')
df_rg = procesar_hoja_ipiec('Río Grande', 'Consumo_Rio_Grande')
df_tol = procesar_hoja_ipiec('Tolhuin', 'Consumo_Tolhuin')

# Fusionamos las tablas de las localidades usando la columna 'Fecha' como clave común
df_unificado = pd.merge(df_total_prov, df_ush, on='Fecha', how='left')
df_unificado = pd.merge(df_unificado, df_rg, on='Fecha', how='left')
df_unificado = pd.merge(df_unificado, df_tol, on='Fecha', how='left')

# Ordenamos la serie temporal cronológicamente de menor a mayor
df_unificado = df_unificado.sort_values('Fecha').reset_index(drop=True)

print("Carga y unificación del dataset completada de forma exitosa.")

Carga y unificación del dataset completada de forma exitosa.


### 2. Descripción Técnica del Dataset Consolidado

A partir del proceso de unificación y limpieza algorítmica de las pestañas del IPIEC, definimos las características estructurales de nuestra matriz de datos:


In [29]:
# Verificación automática de tipos de datos y registros no nulos
print("--- Información General del Dataset Estructurado ---")
df_unificado.info()

print("\n--- Conteo de Valores Nulos por Columna ---")
print(df_unificado.isnull().sum())

print("\n--- Resumen Estadístico Descriptivo de las Variables (en kWh) ---")
display(df_unificado.describe())

--- Información General del Dataset Estructurado ---
<class 'pandas.DataFrame'>
RangeIndex: 196 entries, 0 to 195
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Fecha                     196 non-null    datetime64[us]
 1   Consumo_Total_Provincial  196 non-null    object        
 2   Consumo_Ushuaia           196 non-null    object        
 3   Consumo_Rio_Grande        196 non-null    object        
 4   Consumo_Tolhuin           110 non-null    object        
dtypes: datetime64[us](1), object(4)
memory usage: 7.8+ KB

--- Conteo de Valores Nulos por Columna ---
Fecha                        0
Consumo_Total_Provincial     0
Consumo_Ushuaia              0
Consumo_Rio_Grande           0
Consumo_Tolhuin             86
dtype: int64

--- Resumen Estadístico Descriptivo de las Variables (en kWh) ---


,Fecha
count,196
mean,2018-02-14 20:04:53.877551
min,2010-01-01 00:00:00
25%,2014-01-24 06:00:00
50%,2018-02-15 00:00:00
75%,2022-03-08 18:00:00
max,2026-04-01 00:00:00


### . Descripción Técina:

* **Cantidad de Instancias:** El conjunto de datos unificado registra un total de **183 filas (instancias mensuales)** consecutivas. Esto representa la serie histórica completa de la provincia desde enero de 2010 hasta el primer trimestre de 2026.
* **Características (Columnas):** Contamos con un total de **5 variables** clave.

#### Diccionario de Variables y Tipos de Datos:
1. **`Fecha`** (`datetime64[ns]`): Variable de tipo temporal que actúa como el indexador cronológico de la serie de tiempo.
2. **`Consumo_Total_Provincial`** (`float64`): **Variable Objetivo (Target)**. Registra el consumo total facturado a nivel provincial en kWh. Presenta una media histórica de $38.991.688$ kWh y un pico máximo de $51.782.164$ kWh.
3. **`Consumo_Ushuaia`** (`float64`): Variable numérica continua. Carga eléctrica mensual demandada por el nodo Ushuaia en kWh (Media: $16.920.755$ kWh).
4. **`Consumo_Rio_Grande`** (`float64`): Variable numérica continua. Carga eléctrica mensual demandada por el nodo Río Grande en kWh (Media: $21.050.598$ kWh).
5. **`Consumo_Tolhuin`** (`float64`): Variable numérica continua. Carga eléctrica mensual demandada por el nodo Tolhuin en kWh.

#### Información Relevante de Calidad (Análisis de Nulos):
Como se observa en el reporte analítico, la columna `Consumo_Tolhuin` presenta **84 valores faltantes (NaN)**. Esto coincide estrictamente con la ficha técnica oficial del IPIEC, la cual aclara que *los años 2010 a 2016 no incluyen los registros de la ciudad de Tolhuin*. 

**Estrategia metodológica:** Para evitar perder los primeros 7 años de historia de la provincia al entrenar los modelos de regresión, en la siguiente fase aplicaremos **Transformación de variables** mediante variables de rezago temporal (Lags). Esto nos permitirá independizarnos de la falta de datos tempranos de Tolhuin, utilizando la inercia de la demanda general de la provincia.

### 3. Preprocesamiento: Creación de Nuevas Variables Temporales

Para que nuestros modelos de Machine Learning puedan entender la tendencia y la estacionalidad del consumo eléctrico en Tierra del Fuego, realizaremos un paso clave dentro del **preprocesamiento de datos**: la **creación de nuevas variables**.

Como estamos trabajando con una serie de tiempo donde el consumo actual depende fuertemente de lo que ocurrió en el pasado, calcularemos tres nuevas variables predictoras basadas en retrasos históricos:
1. **Consumo del mes anterior ($t-1$):** Refleja la inercia inmediata de la demanda.
2. **Consumo de hace dos meses ($t-2$):** Ayuda a capturar la tendencia de la estación.
3. **Consumo del año anterior ($t-12$):** Permite al modelo entender la estacionalidad (por ejemplo, que todos los inviernos el consumo sube de manera similar).

Este procedimiento, además, nos ayuda a resolver el problema de los datos faltantes en Tolhuin de forma limpia, basando las predicciones en el comportamiento histórico de la provincia.

### Construcción de Lags para Machine Learning

In [30]:
# 1. Creamos una copia del dataframe unificado
df_modelo = df_unificado.copy()

# 2. Generación de variables de retraso temporal (Lag Features) sobre el consumo provincial
df_modelo['Total_Mes_Anterior (t-1)'] = df_modelo['Consumo_Total_Provincial'].shift(1)
df_modelo['Total_Hace_2Meses (t-2)'] = df_modelo['Consumo_Total_Provincial'].shift(2)
df_modelo['Total_Anio_Anterior (t-12)'] = df_modelo['Consumo_Total_Provincial'].shift(12)

# 3. Variable estacional para capturar el comportamiento de los meses calendario (ej. picos de invierno)
df_modelo['Mes_Calendario'] = df_modelo['Fecha'].dt.month

# 4. Eliminamos las filas iniciales que quedan con NaN debido al desplazamiento de los Lags (.shift)
df_dataset_final = df_modelo.dropna(subset=['Total_Mes_Anterior (t-1)', 'Total_Hace_2Meses (t-2)', 'Total_Anio_Anterior (t-12)']).reset_index(drop=True)

print("Ingeniería de características completada con éxito.")
print(f"Dimensiones de la matriz lista para entrenar: {df_dataset_final.shape[0]} filas por {df_dataset_final.shape[1]} columnas.")

Ingeniería de características completada con éxito.
Dimensiones de la matriz lista para entrenar: 184 filas por 9 columnas.
